📊 Descrição das Variáveis
O dataset contém variáveis meteorológicas diárias coletadas em diferentes estações da Austrália. As principais variáveis incluem:

Date: Data da observação
Location: Nome da estação meteorológica

🌡️ Temperatura

MinTemp / MaxTemp: Temperatura mínima e máxima do dia (°C);
Temp9am / Temp3pm: Temperatura às 9h e às 15h

🌧️ Precipitação

Rainfall: Volume de chuva no dia (mm);
RainToday: Indica se choveu no dia atual (≥ 1 mm);
RainTomorrow: Variável alvo, indicando se choverá no dia seguinte

💨 Vento

WindGustDir / WindGustSpeed: Direção e velocidade da rajada mais forte;
WindDir9am / WindDir3pm: Direção do vento às 9h e 15h;
WindSpeed9am / WindSpeed3pm: Velocidade média do vento

💧 Umidade e Pressão

Humidity9am / Humidity3pm: Umidade relativa (%);
Pressure9am / Pressure3pm: Pressão atmosférica (hPa)

☁️ Nebulosidade

Cloud9am / Cloud3pm: Cobertura de nuvens (0 a 8, em oitavos do céu)

☀️ Outras variáveis

Sunshine: Horas de sol no dia;
Evaporation: Evaporação medida (mm)

In [2]:
# Conexão com Arquivos no Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Leitura do Dataset Selecionado
import pandas as pd
df = pd.read_csv("weatherAUS_Original.csv")

In [2]:
# Validação da Leitura com informações iniciais
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [3]:
# Limpar as células vazias do meu Target
df = df.dropna(subset=["RainTomorrow"])

In [4]:
# Análise de valores ausentes nas variáveis do dataset, com o objetivo de identificar colunas com alto percentual de dados faltantes e definir a estratégia de tratamento mais adequada.
na_summary = pd.DataFrame({
    "NA_count": df.isna().sum(),
    "NA_percent": df.isna().mean() * 100
}).sort_values("NA_percent", ascending=False)

na_summary

,NA_count,NA_percent
Sunshine,67816,47.692924
Evaporation,60843,42.789026
Cloud3pm,57094,40.152469
Cloud9am,53657,37.735332
Pressure9am,14014,9.855619
Pressure3pm,13981,9.832411
WindDir9am,10013,7.041838
WindGustDir,9330,6.561504
WindGustSpeed,9270,6.519308
WindDir3pm,3778,2.656952


In [5]:
# Devido a quantidade de dados vazios nas colunas Sunshine, Cloud3pm e Cloud9am e devido a informação dessas variáveis não serem tão relevantes para previsão de chuvas elas foram deletadas do dataset.
# A variável Evaporation foi mantida no dataset, apesar do elevado percentual de valores ausentes (~43%), devido à sua relevância física no contexto climático.
cols_to_drop = ["Sunshine", "Cloud3pm", "Cloud9am"]
df = df.drop(columns=cols_to_drop)

As colunas textuais do dataset representam variáveis categóricas, como localização da estação meteorológica, direção do vento e ocorrência de chuva no dia atual. Como algoritmos de machine learning trabalham com valores numéricos, essas variáveis precisam ser convertidas para uma representação numérica antes do treinamento.

No caso de variáveis categóricas sem ordem natural, como Location, WindGustDir, WindDir9am e WindDir3pm, não é adequado substituir os textos por números inteiros simples, pois isso poderia induzir o modelo a interpretar uma ordem inexistente entre as categorias. Para resolver esse problema, foi utilizada a técnica de one-hot encoding, que cria uma nova coluna binária para cada categoria observada.

Assim, cada valor textual passa a ser representado por colunas contendo 0 ou 1, permitindo que os algoritmos utilizem essas informações corretamente.

In [6]:
# colunas categóricas mais adequadas para encoding
cat_cols = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
# Para fins de compreensão do processo de transformação das variáveis categóricas, foi utilizado pd.get_dummies() em uma etapa exploratória.
# Contudo, no treinamento final dos modelos, optou-se por realizar essa transformação dentro do pipeline com OneHotEncoder, conforme recomendado pelas boas práticas de machine learning.

In [7]:
# Os dados foram separados em variáveis independentes (X) e variável alvo (y). A variável RainTomorrow foi definida como target do modelo, enquanto as demais variáveis foram utilizadas como entrada para treinamento.
X = df.drop("RainTomorrow", axis=1)
y = df["RainTomorrow"].map({"No": 0, "Yes": 1})

**`PRIMEIRO TESTE: Sem retirar os NaN da Coluna EVAPORATION`**

In [8]:
# Os dados foram divididos em conjuntos de treino e teste utilizando a técnica de holdout, sendo 80% destinados ao treinamento e 20% à avaliação do modelo. Foi utilizada estratificação para manter a proporção das classes.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
# As variáveis de entrada foram separadas em dois grupos: numéricas e categóricas. Essa separação é necessária para que cada tipo de variável receba o tratamento adequado dentro do pipeline de pré-processamento.
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

In [10]:
# Para as variáveis numéricas, foi criado um pipeline com imputação dos valores ausentes pela mediana e posterior padronização por meio do StandardScaler.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
# Para as variáveis categóricas, foi utilizado um pipeline com imputação pelo valor mais frequente e transformação por OneHotEncoder, permitindo representar as categorias em formato numérico.
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [12]:
# O pré-processamento foi consolidado em um ColumnTransformer, permitindo aplicar diferentes transformações para variáveis numéricas e categóricas dentro de uma única estrutura.
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

KNN

In [ ]:
# O pipeline final foi composto pelo bloco de pré-processamento e pelo algoritmo KNN, permitindo automatizar as etapas de transformação dos dados e treinamento do modelo.
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

In [ ]:
knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy: 0.8413094693906256

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.94      0.90     22064
           1       0.72      0.48      0.58      6375

    accuracy                           0.84     28439
   macro avg       0.79      0.71      0.74     28439
weighted avg       0.83      0.84      0.83     28439


Confusion Matrix:

[[20838  1226]
 [ 3287  3088]]


O modelo KNN apresentou acurácia de 84,13% no conjunto de teste. Entretanto, a análise detalhada mostra que seu desempenho foi significativamente melhor para a classe 0 (dias sem chuva) do que para a classe 1 (dias com chuva).

Para a classe 0, o modelo obteve precision de 0,86, recall de 0,94 e F1-score de 0,90, indicando excelente capacidade de identificar corretamente dias sem precipitação.

Para a classe 1, a precision foi de 0,72, mas o recall caiu para 0,48, mostrando que o modelo identificou menos da metade dos dias em que realmente houve chuva. O F1-score dessa classe foi 0,58, reforçando a limitação do modelo em detectar eventos de chuva.

A matriz de confusão confirmou esse comportamento, evidenciando um número elevado de falsos negativos (dias com chuva classificados como sem chuva). Assim, embora a acurácia global seja satisfatória, o desempenho na classe minoritária ainda precisa ser melhorado.

**`Segundo TESTE: Retirando a Coluna EVAPORATION`**

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/weatherAUS_Original.csv")
df = df.dropna(subset=["RainTomorrow"])

cols_to_drop = ["Sunshine", "Cloud3pm", "Cloud9am","Evaporation"]

df = df.drop(columns=cols_to_drop)
X = df.drop("RainTomorrow", axis=1)
y = df["RainTomorrow"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy: 0.8405358838215127

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.94      0.90     22064
           1       0.71      0.48      0.58      6375

    accuracy                           0.84     28439
   macro avg       0.79      0.71      0.74     28439
weighted avg       0.83      0.84      0.83     28439


Confusion Matrix:

[[20819  1245]
 [ 3290  3085]]


**`Terceiro TESTE: Retirando os NaN da Coluna EVAPORATION`**

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/weatherAUS_Original.csv")
df = df.dropna(subset=["RainTomorrow"])

df = df.dropna(subset=["Evaporation"])

cols_to_drop = ["Sunshine", "Cloud3pm", "Cloud9am"]
df = df.drop(columns=cols_to_drop)
X = df.drop("RainTomorrow", axis=1)
y = df["RainTomorrow"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)
y_pred_knn = knn_pipeline.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_knn))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy: 0.8401352181929932

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.94      0.90     12698
           1       0.70      0.47      0.57      3572

    accuracy                           0.84     16270
   macro avg       0.78      0.71      0.73     16270
weighted avg       0.83      0.84      0.83     16270


Confusion Matrix:

[[11978   720]
 [ 1881  1691]]


Foram realizados testes comparativos considerando três abordagens para a variável Evaporation: (i) manutenção da variável com imputação de valores ausentes, (ii) remoção completa da variável e (iii) remoção das instâncias com valores ausentes.

Os resultados obtidos indicaram que a presença ou ausência da variável Evaporation não impactou significativamente o desempenho do modelo, com valores de acurácia e métricas de classificação muito semelhantes entre os cenários.

Por outro lado, a remoção de instâncias com valores ausentes resultou em uma redução significativa do tamanho do dataset, sem ganho de desempenho.

Dessa forma, optou-se por manter a variável e tratar seus valores ausentes por meio de imputação no pipeline, preservando a maior quantidade possível de dados.

🌳 Decision Tree

In [29]:
# Foi testado o algoritmo de Árvore de Decisão, que constrói um modelo baseado em regras hierárquicas de decisão. Esse algoritmo busca dividir os dados em grupos cada vez mais homogêneos em relação à variável alvo.

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/weatherAUS_Original.csv")
df = df.dropna(subset=["RainTomorrow"])

cols_to_drop = ["Sunshine", "Cloud3pm", "Cloud9am"]
df = df.drop(columns=cols_to_drop)
X = df.drop("RainTomorrow", axis=1)
y = df["RainTomorrow"].map({"No": 0, "Yes": 1})

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

from sklearn.tree import DecisionTreeClassifier

tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

In [30]:
# O pipeline foi treinado utilizando o conjunto de treino. Nessa etapa, o modelo aprende simultaneamente o pré-processamento dos dados e as regras de decisão da árvore.
tree_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MinTemp', 'MaxTemp',
                                                   'Rainfall', 'Evaporation',
                                                   'WindGustSpeed',
                                                   'WindSpeed9am',
                                                   'WindSpeed3pm',
                                                   'Humidity9am', 'Humidity3pm',
                                                   'Pressure9am', 'Pressure3pm',
                                                   'Temp9am', 'Temp3pm']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Date', 'Location',
                                                   'WindGustDir', 'WindDir9am',
                                                   'WindDir3pm',
                                                   'RainToday'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [31]:
# Após o treinamento, o modelo foi utilizado para realizar predições no conjunto de teste, que contém dados não vistos durante o treinamento.
y_pred_tree = tree_pipeline.predict(X_test)

In [32]:
# O desempenho do modelo foi avaliado por meio de métricas de classificação, incluindo acurácia, precision, recall, F1-score e matriz de confusão.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_tree))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_tree))

Accuracy: 0.812757129294279

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.89      0.88     22064
           1       0.59      0.54      0.56      6375

    accuracy                           0.81     28439
   macro avg       0.73      0.71      0.72     28439
weighted avg       0.81      0.81      0.81     28439


Confusion Matrix:

[[19687  2377]
 [ 2948  3427]]


O modelo de Árvore de Decisão apresentou acurácia de aproximadamente 81%, inferior ao modelo KNN. No entanto, observou-se uma melhora significativa no recall da classe 1 (dias com chuva), que passou de aproximadamente 48% para 54%.

Isso indica que o modelo conseguiu identificar um número maior de eventos de chuva, reduzindo a quantidade de falsos negativos.

Por outro lado, houve um aumento no número de falsos positivos, refletido na redução da precisão da classe 1.

Dessa forma, a Árvore de Decisão apresentou um comportamento mais equilibrado entre as classes, sendo mais adequada para cenários em que a detecção de eventos de chuva é mais importante do que a acurácia global.

Naive Bayes

Para o modelo Naive Bayes, foi necessário ajustar o OneHotEncoder para gerar uma matriz densa, uma vez que o classificador GaussianNB não suporta matrizes esparsas. Dessa forma, foi possível integrar corretamente o algoritmo ao pipeline de pré-processamento.

In [42]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [43]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [44]:
from sklearn.naive_bayes import GaussianNB

nb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

In [45]:
nb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MinTemp', 'MaxTemp',
                                                   'Rainfall', 'Evaporation',
                                                   'WindGustSpeed',
                                                   'WindSpeed9am',
                                                   'WindSpeed3pm',
                                                   'Humidity9am', 'Humidity3pm',
                                                   'Pressure9am', 'Pressure3pm',
                                                   'Temp9am', 'Temp3pm']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Date', 'Location',
                                                   'WindGustDir', 'WindDir9am',
                                                   'WindDir3pm',
                                                   'RainToday'])])),
                ('classifier', GaussianNB())])

In [46]:
y_pred_nb = nb_pipeline.predict(X_test)

In [47]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.46102183621083725

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.35      0.50     22064
           1       0.27      0.85      0.41      6375

    accuracy                           0.46     28439
   macro avg       0.58      0.60      0.46     28439
weighted avg       0.75      0.46      0.48     28439


Confusion Matrix:

[[ 7695 14369]
 [  959  5416]]


O modelo Naive Bayes apresentou desempenho significativamente inferior aos demais modelos, com acurácia de aproximadamente 46%.

Observou-se um comportamento fortemente enviesado para a classe “chuva”, resultando em um recall elevado (85%), mas com baixa precisão (27%), indicando grande quantidade de falsos positivos.

Esse comportamento pode ser explicado pela suposição de independência entre as variáveis feita pelo algoritmo, o que não se verifica neste dataset, onde diversas variáveis meteorológicas são correlacionadas.

Dessa forma, embora o modelo seja capaz de identificar a maioria dos eventos de chuva, ele apresenta baixa confiabilidade nas previsões, tornando-o menos adequado para o problema proposto.

SVM

In [13]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
# Foi testado o algoritmo Support Vector Machine (SVM), que busca encontrar um hiperplano que melhor separa as classes, maximizando a margem entre elas. Devido ao alto custo computacional do SVM tradicional, foi utilizada a implementação LinearSVC, mais eficiente para grandes volumes de dados.
from sklearn.svm import LinearSVC

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(random_state=42, max_iter=5000))
])

In [15]:
# O pipeline foi treinado utilizando o conjunto de treino, permitindo que o modelo aprendesse os padrões que melhor separam as classes de dias com e sem chuva.
svm_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [16]:
# Após o treinamento, o modelo SVM foi aplicado ao conjunto de teste para gerar as previsões da variável alvo.
y_pred_svm = svm_pipeline.predict(X_test)

In [17]:
# O desempenho do modelo foi avaliado por meio de métricas de classificação, incluindo acurácia, precision, recall, F1-score e matriz de confusão, permitindo compará-lo com os demais modelos testados.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))

Accuracy: 0.8527726010056612

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.95      0.91     22064
           1       0.74      0.53      0.62      6375

    accuracy                           0.85     28439
   macro avg       0.81      0.74      0.76     28439
weighted avg       0.84      0.85      0.84     28439


Confusion Matrix:

[[20851  1213]
 [ 2974  3401]]


O modelo SVM (LinearSVC) apresentou o melhor desempenho geral entre os algoritmos testados, com acurácia de aproximadamente 85%.

Para a classe de dias com chuva, o modelo obteve precision de 0,74, recall de 0,53 e F1-score de 0,62, indicando um bom equilíbrio entre a capacidade de identificar eventos de chuva e evitar falsos positivos.

Em comparação com os demais modelos, o SVM demonstrou o melhor compromisso entre desempenho global e qualidade das previsões para a classe minoritária, sendo considerado o modelo mais adequado para o problema proposto.

Foram avaliados quatro modelos de classificação: KNN, Árvore de Decisão, Naive Bayes e SVM.

O modelo KNN apresentou boa acurácia, porém baixo recall para a classe de chuva, indicando dificuldade em identificar corretamente esses eventos.

A Árvore de Decisão apresentou melhora no recall, tornando-se mais equilibrada, porém com leve redução na acurácia.

O Naive Bayes apresentou comportamento instável, com alto recall, porém baixa precisão e acurácia, sendo pouco adequado para o problema.

O SVM apresentou o melhor desempenho geral, combinando alta acurácia com bom equilíbrio entre precision e recall, especialmente para a classe minoritária.

Dessa forma, o modelo SVM foi selecionado como a melhor abordagem entre as avaliadas.

Foi realizada a otimização do modelo SVM utilizando GridSearchCV com validação cruzada, testando diferentes valores do hiperparâmetro C e avaliando o impacto do balanceamento de classes por meio do parâmetro class_weight.

In [18]:
# Para melhorar o desempenho do modelo SVM, foi realizada a otimização de hiperparâmetros por meio de GridSearchCV com validação cruzada. O hiperparâmetro ajustado foi o parâmetro C, que controla o equilíbrio entre margem de separação e penalização de erros de classificação.
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(random_state=42, max_iter=10000))
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and can

In [19]:
# Após o processo de validação cruzada, foi identificado o melhor valor do hiperparâmetro C para o modelo SVM.
print("Melhor parâmetro:", grid_svm.best_params_)
print("Melhor score médio na validação cruzada:", grid_svm.best_score_)

Melhor parâmetro: {'classifier__C': 10}
Melhor score médio na validação cruzada: 0.7573567448024539


In [20]:
# O melhor modelo encontrado foi então avaliado no conjunto de teste, permitindo comparar seu desempenho com a versão inicial do SVM.
best_svm = grid_svm.best_estimator_

y_pred_svm_tuned = best_svm.predict(X_test)

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_svm_tuned))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm_tuned))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm_tuned))

Accuracy: 0.8528429269664897

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.94      0.91     22064
           1       0.74      0.54      0.62      6375

    accuracy                           0.85     28439
   macro avg       0.81      0.74      0.76     28439
weighted avg       0.84      0.85      0.84     28439


Confusion Matrix:

[[20836  1228]
 [ 2957  3418]]


Foi realizada a otimização do hiperparâmetro C do modelo SVM utilizando GridSearchCV com validação cruzada. O melhor valor encontrado foi C = 10.

No entanto, a melhoria no desempenho foi marginal em relação ao modelo original, indicando que o SVM já apresentava desempenho próximo do ótimo com os parâmetros padrão.

Esse resultado sugere que o principal desafio do problema não está na escolha do hiperparâmetro C, mas sim em características intrínsecas do dataset, como o desbalanceamento entre as classes.

Para lidar com o desbalanceamento das classes, foi incluído o parâmetro class_weight no modelo SVM. A opção "balanced" ajusta automaticamente os pesos das classes inversamente proporcionais às suas frequências, fazendo com que erros na classe minoritária (dias com chuva) sejam mais penalizados durante o treinamento.

In [22]:
# Para melhorar o desempenho do modelo SVM, foi realizada a otimização de hiperparâmetros por meio de GridSearchCV com validação cruzada. O hiperparâmetro ajustado foi o parâmetro C, que controla o equilíbrio entre margem de separação e penalização de erros de classificação.
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(random_state=42, max_iter=10000))
])

param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"]
}

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 1, ...], 'classifier__class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the sc

In [23]:
# Após o processo de validação cruzada, foi identificado o melhor valor do hiperparâmetro C para o modelo SVM.
print("Melhor parâmetro:", grid_svm.best_params_)
print("Melhor score médio na validação cruzada:", grid_svm.best_score_)

Melhor parâmetro: {'classifier__C': 10, 'classifier__class_weight': None}
Melhor score médio na validação cruzada: 0.7573567448024539


In [24]:
# O melhor modelo encontrado foi então avaliado no conjunto de teste, permitindo comparar seu desempenho com a versão inicial do SVM.
best_svm = grid_svm.best_estimator_

y_pred_svm_tuned = best_svm.predict(X_test)

In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_svm_tuned))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm_tuned))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm_tuned))

Accuracy: 0.8528429269664897

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.94      0.91     22064
           1       0.74      0.54      0.62      6375

    accuracy                           0.85     28439
   macro avg       0.81      0.74      0.76     28439
weighted avg       0.84      0.85      0.84     28439


Confusion Matrix:

[[20836  1228]
 [ 2957  3418]]


Foi testada também a inclusão do parâmetro class_weight="balanced" no modelo SVM, com o objetivo de compensar o desbalanceamento entre as classes. No entanto, o processo de otimização por validação cruzada indicou que a melhor configuração continuou sendo aquela sem balanceamento (class_weight=None), mantendo desempenho praticamente idêntico ao modelo anterior.

Dentre os modelos avaliados, o SVM (LinearSVC) apresentou o melhor desempenho geral, combinando alta acurácia com bom equilíbrio entre precision e recall, especialmente para a classe minoritária (dias com chuva).

Diferentemente dos demais modelos, o SVM conseguiu manter um nível elevado de precisão ao mesmo tempo em que apresentou capacidade adequada de detecção de eventos de chuva, resultando no melhor F1-score para essa classe.

Dessa forma, o modelo SVM foi selecionado como a abordagem mais adequada para o problema proposto, sendo o escolhido para utilização em produção.

In [26]:
import joblib

joblib.dump(best_svm, "modelo_svm.pkl")

['modelo_svm.pkl']